# Visualize Results

This notebook contains the code for generating all the plots used in the paper

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
currency_symbols = {
    'US': '$', 'ES': '€', 'GB': '£', 'CN': '¥', 'JP': '¥', 'IN': '₹', 'DE': '€', 'FR': '€', 'SG':'$','BR': 'R$'
    # Add more country codes and symbols as needed
}
import os
os.chdir("../")

In [2]:
from utils.visualization_utils import region_policy_scatter_plot_panel, best_policy_cost_breakdown_plot_panel, shorten_policy_string
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math


## Brazil

In [3]:
# results = pd.read_csv("simulation_results/test_result_20211126_004503.csv")
results_BR = pd.read_csv("simulation_results/test_result_BR_20240701_151024.csv")
results_DE = pd.read_csv("simulation_results/test_result_DE_20240701_151038.csv")
# results_USFL = pd.read_csv("simulation_results/test_result_US-FL_20240701_104146.csv")
results_USNY = pd.read_csv("simulation_results/test_result_US-NY_20240701_150957.csv")
# results_SG = pd.read_csv("simulation_results/test_result_SG_20240701_110548.csv")
results_ES = pd.read_csv("simulation_results/test_result_ES_20240701_151011.csv")

results = pd.concat([results_BR,results_DE,results_USNY,results_ES],axis=0)
results

,Unnamed: 0,country,start_date,policy_length,policy,st_economic_costs,st_economic_costs_lb,st_economic_costs_ub,lt_economic_costs,d_costs,...,num_deaths_ub,hospitalization_days,hospitalization_days_lb,hospitalization_days_ub,icu_days,icu_days_lb,icu_days_ub,ventilated_days,ventilated_days_lb,ventilated_days_ub
0,0,BR,2020-03-15,3,actual,2.829457e+10,2.829457e+10,2.829457e+10,0,1.861698e+11,...,NaN,1.432788e+06,NaN,NaN,52488.529412,NaN,NaN,297435,NaN,NaN
1,1,BR,2020-03-15,3,No_Measure-No_Measure-No_Measure,2.389453e+10,1.785228e+10,2.993679e+10,0,2.415088e+12,...,608205.0,1.457128e+07,1.457128e+07,1.457128e+07,567910.411765,567910.411765,567910.411765,3218159,3218159.0,3218159.0
2,2,BR,2020-03-15,3,No_Measure-No_Measure-Restrict_Mass_Gatherings,2.369894e+10,1.896695e+10,2.843093e+10,0,1.898793e+12,...,478184.0,1.210552e+07,1.210552e+07,1.210552e+07,467185.764706,467185.764706,467185.764706,2647386,2647386.0,2647386.0
3,3,BR,2020-03-15,3,No_Measure-No_Measure-Restrict_Mass_Gatherings...,2.836223e+10,2.631452e+10,3.040994e+10,0,8.401634e+11,...,211583.0,6.898641e+06,6.898641e+06,6.898641e+06,255551.647059,255551.647059,255551.647059,1448126,1448126.0,1448126.0
4,4,BR,2020-03-15,3,No_Measure-No_Measure-Authorize_Schools_but_Re...,2.414647e+10,2.012921e+10,2.816374e+10,0,1.617326e+12,...,407300.0,1.076307e+07,1.076307e+07,1.076307e+07,412398.882353,412398.882353,412398.882353,2336927,2336927.0,2336927.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,212,ES,2020-03-15,3,Lockdown-Lockdown-Restrict_Mass_Gatherings,5.550278e+10,5.549502e+10,5.551055e+10,0,3.897523e+12,...,25789.0,4.468016e+05,4.468016e+05,4.468016e+05,15214.411765,15214.411765,15214.411765,86215,86215.0,86215.0
213,213,ES,2020-03-15,3,Lockdown-Lockdown-Restrict_Mass_Gatherings_and...,6.836994e+10,6.836567e+10,6.837422e+10,0,2.651887e+12,...,17547.0,3.505490e+05,3.505490e+05,3.505490e+05,11199.000000,11199.000000,11199.000000,63461,63461.0,63461.0
214,214,ES,2020-03-15,3,Lockdown-Lockdown-Authorize_Schools_but_Restri...,5.808963e+10,5.808314e+10,5.809612e+10,0,3.457255e+12,...,22876.0,4.132795e+05,4.132795e+05,4.132795e+05,13811.470588,13811.470588,13811.470588,78265,78265.0,78265.0
215,215,ES,2020-03-15,3,Lockdown-Lockdown-Restrict_Mass_Gatherings_and...,7.023485e+10,7.023139e+10,7.023831e+10,0,2.299519e+12,...,15216.0,3.189320e+05,3.189320e+05,3.189320e+05,9927.000000,9927.000000,9927.000000,56253,56253.0,56253.0


In [4]:
results["start_date"] = pd.to_datetime(results.start_date)
results['life_costs'] = results.d_costs + results.h_costs + results.mh_costs
results["short_policy_name"] = [
    pname if pname == "actual" else shorten_policy_string(pname)
    for pname in results["policy"]
]
results['is_actual'] = ["Actual" if pn == 'actual'  else "hypothetical"  for pn in results['short_policy_name'] ]

In [9]:
def shorten_policy_string(pname):
    policies = pname.split("-")
    DICT_POLICY_CODE = {
        'No_Measure': "1",
        'Restrict_Mass_Gatherings': "2",
        'Authorize_Schools_but_Restrict_Mass_Gatherings_and_Others': "3",
        # 'Mass_Gatherings_Authorized_But_Others_Restricted': "3",
        'Restrict_Mass_Gatherings_and_Schools': "4",
        'Restrict_Mass_Gatherings_and_Schools_and_Others': "5",
        'Lockdown': "6"
    }
    short_name = '-'.join([DICT_POLICY_CODE[pol] for pol in policies])
    return short_name

# Currency symbol mapping
currency_symbols = {
    'US': '$', 'ES': '€', 'GB': '£', 'CN': '¥', 'JP': '¥', 'IN': '₹', 'DE': '€', 'FR': '€',
    # Add more country codes and symbols as needed
}
name_codes = {
    'US-NY': 'New York, United States', 'ES': 'Spain', 'DE': 'Germany', 'BR': 'Brazil'
    # Add more country codes and symbols as needed
}

cost_colors = {
    'st_economic_costs': '#636EFA',  # blue
    'd_costs': '#EF553B',  # red
    'h_costs': '#00CC96',  # green
    'mh_costs': '#AB63FA'  # purple
}
def get_currency_symbol(country_code):
    # Extract the ISO country code (first two characters)
    iso_code = country_code.split('-')[0]
    return currency_symbols.get(iso_code, '$')  # Default to '$' if not found


def region_policy_scatter_plot_panel(results: pd.DataFrame, start_date: str = "3/15/2020", 
                                     y_val: str = 'num_deaths', width: int = 600, height: int = 600):
    regions = results['country'].unique()
    num_regions = len(regions)
    num_rows = math.ceil(num_regions / 2)

    
    # Create a subplot grid with two columns
    fig = make_subplots(rows=num_rows, cols=2, shared_xaxes=False, shared_yaxes=False, 
                        subplot_titles=[name_codes[x] for x in regions], vertical_spacing=0.1, horizontal_spacing=0.1)
    
    for i, region in enumerate(regions):
        df = results.query("start_date == @start_date and country == @region")
        if y_val == 'life_costs':
            df['life_costs'] = df.d_costs + df.h_costs + df.mh_costs
            df['life_costs_lb'] = df.d_costs_lb + df.h_costs + df.mh_costs_lb
            df['life_costs_ub'] = df.d_costs_ub + df.h_costs + df.mh_costs_ub
        df['st_economic_costs_lerr'] = df['st_economic_costs'] - df['st_economic_costs_lb']
        df['st_economic_costs_uerr'] = df['st_economic_costs_ub'] - df['st_economic_costs']
        df[f'{y_val}_lerr'] = df[y_val] - df[f'{y_val}_lb']
        df[f'{y_val}_uerr'] = df[f'{y_val}_ub'] - df[y_val]

        # Calculate average strength and add it as a column
        df['avg_strength'] = df['short_policy_name'].apply(lambda x: sum(map(int, x.split('-'))) / 3 if x != 'actual' else None)

        y_val_name = 'Number of Deaths' if y_val == 'num_deaths' else \
            'Humanitarian Costs' if y_val == 'life_costs' else y_val

        # Get the currency symbol for the region
        country_code = df['country'].iloc[0]
        currency_symbol = get_currency_symbol(country_code)
        
        scatter = px.scatter(df, x='st_economic_costs', y=y_val, color='avg_strength',
                             error_x='st_economic_costs_uerr', error_x_minus='st_economic_costs_lerr',
                             error_y=f'{y_val}_uerr', error_y_minus=f'{y_val}_lerr', log_x=False, log_y=True, 
                             hover_name="short_policy_name", hover_data=["num_deaths", "num_cases", "mh_costs", "h_costs", "avg_strength"],
                             labels={'st_economic_costs': 'Economic Costs', y_val: y_val_name, "avg_strength": "Average Strength"},
                             template="plotly_white", color_continuous_scale=px.colors.sequential.Reds)

        # Add traces to the subplot
        for trace in scatter.data:
            fig.add_trace(trace, row=(i//2) + 1, col=(i % 2) + 1)
        
        # Highlight the policy "6-6-6" without adding it to the legend multiple times
        highlight_policy = df[df['short_policy_name'] == '6-6-6']
        if not highlight_policy.empty:
            fig.add_trace(go.Scatter(
                x=highlight_policy['st_economic_costs'],
                y=highlight_policy[y_val],
                mode='markers+text',
                marker=dict(color='red', size=12, symbol='diamond'),
                showlegend=False,
                text=["6-6-6"],
                textposition="top center",
                hoverinfo='skip'
            ), row=(i//2) + 1, col=(i % 2) + 1)

        # Highlight the "actual" policy without adding it to the legend multiple times
        actual_policy = df[df['short_policy_name'] == 'actual']
        if not actual_policy.empty:
            fig.add_trace(go.Scatter(
                x=actual_policy['st_economic_costs'],
                y=actual_policy[y_val],
                mode='markers+text',
                marker=dict(color='blue', size=12, symbol='star'),
                showlegend=False,
                text=["Actual"],
                textposition="top center",
                hoverinfo='skip'
            ), row=(i//2) + 1, col=(i % 2) + 1)

        # Update axes with the correct currency symbol and log scale for y-axis
        fig.update_xaxes(title_text='Economic Costs', tickprefix=currency_symbol, row=(i//2) + 1, col=(i % 2) + 1)
        fig.update_yaxes(title_text=y_val_name, type='log', row=(i//2) + 1, col=(i % 2) + 1)
        if y_val == 'life_costs':
            fig.update_yaxes(tickprefix=currency_symbol, type='log', row=(i//2) + 1, col=(i % 2) + 1)

    # Add legend items for the special points
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(color='red', size=12, symbol='diamond'),
        name="Most Severe Restrictions"
    ))
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(color='blue', size=12, symbol='star'),
        name="Actual Policy"
    ))

    # Update layout
    fig.update_layout(
        legend=dict(title='Reference Policies', orientation='h', y=1.05, yanchor='bottom', x=0.5, xanchor='center', traceorder='normal'),
        coloraxis_colorbar=dict(title='Average<br>Strength'),
        margin=dict(l=40, r=40, t=80, b=40),
        plot_bgcolor='rgba(0,0,0,0)',
        width=width * 2,
        height=height * num_rows
    )

    return fig
region_policy_scatter_plot_panel(results, start_date = "3/15/2020")
# region_policy_scatter_plot_panel(results, region_name = "BR",start_date = "3/15/2020")

/var/folders/ht/nmdv07xs10n2t90rd4sl1ww80000gn/T/ipykernel_53107/4074751375.py:49: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/var/folders/ht/nmdv07xs10n2t90rd4sl1ww80000gn/T/ipykernel_53107/4074751375.py:54: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/ht/nmdv07xs10n2t90rd4sl1ww80000gn/T/ipykernel_53107/4074751375.py:55: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda

In [9]:
fig = region_policy_scatter_plot_panel(results, start_date = "3/15/2020", y_val = "life_costs")
fig.write_image("simulation_results/scatter_plot.pdf")

/Users/michaelli/Documents/GitHub/THEMIS/utils/visualization_utils.py:57: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/Users/michaelli/Documents/GitHub/THEMIS/utils/visualization_utils.py:59: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/Users/michaelli/Documents/GitHub/THEMIS/utils/visualization_utils.py:60: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/u

In [8]:
results['total_cost'] = results.life_costs + results.st_economic_costs
fig = best_policy_cost_breakdown_plot_panel(results, n = 20, start_date = "3/15/2020")
fig.write_image("simulation_results/best_policies.pdf")

/Users/michaelli/Documents/GitHub/THEMIS/utils/visualization_utils.py:320: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/Users/michaelli/Documents/GitHub/THEMIS/utils/visualization_utils.py:320: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/Users/michaelli/Documents/GitHub/THEMIS/utils/visualization_utils.py:320: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/Users/michae